# 03 – Spatial 5-fold Cross-Validation

Evaluates spatial generalisation by withholding a different set of gauging
stations in each fold.  The model is trained on the remaining stations and
evaluated on the held-out ones, using the same full network for routing.

Fold assignments are read from pre-defined CSV files in which a `valsites`
column flags validation stations (`valsites == 1`).

**Prerequisites:** `01_data_preparation.ipynb`

**Outputs** (in `<output_dir>/`):
- `fold<k>_best.pt` – best checkpoint per fold
- `fold<k>_last.pt` – last-epoch checkpoint per fold
- `5fold_spatial_results.json` – losses and R² per fold and epoch


In [ ]:
import os, random, json, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import SequentialLR, LinearLR, StepLR
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

from sparrow.model import SPARROW, ParamGenerator
from sparrow.utils import setup_seed, gpu_align, r2_torch, r2_logspace_torch

# ── User settings ─────────────────────────────────────────────────────────
prepared_dir = './outputs/prepared/'
output_dir   = './outputs/spatial_cv/'
os.makedirs(output_dir, exist_ok=True)

NUM_EPOCHS = 150
SEED       = 42
setup_seed(SEED)


In [ ]:
with open(os.path.join(prepared_dir, 'prepared_data.pkl'), 'rb') as f:
    prep = pickle.load(f)

input_data_huc12   = prep['input_data_huc12']
dlvdsgn_array      = prep['dlvdsgn_array']
source_columns     = prep['source_columns'];  delivery_columns  = prep['delivery_columns']
stream_columns     = prep['stream_columns'];  reservoir_columns = prep['reservoir_columns']
input_columns      = prep['input_columns']
input_columns_strm = prep['input_columns_strm']
input_columns_res  = prep['input_columns_res']
scaler = prep['scaler']; scaler_strm = prep['scaler_strm']; scaler_res = prep['scaler_res']
years  = prep['years']

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dlvdsgn = torch.tensor(dlvdsgn_array, dtype=torch.float32).to(device)
print(f"Device: {device}")

ns = len(source_columns); nd = len(delivery_columns)


In [ ]:
# Build yearly DataLoaders
yearly_loaders = []
for year in years:
    yd = input_data_huc12[input_data_huc12['Year'] == year].copy()
    X_sc      = torch.tensor(scaler.transform(yd[input_columns].values),
                             dtype=torch.float32, device=device)
    X_sc_strm = torch.tensor(scaler_strm.transform(yd[input_columns_strm].values),
                             dtype=torch.float32, device=device)
    X_sc_res  = torch.tensor(scaler_res.transform(yd[input_columns_res].values),
                             dtype=torch.float32, device=device)
    obs = torch.tensor(yd['depvar'].values, dtype=torch.float32, device=device)
    obs[obs == 0] = float('nan')
    ds = TensorDataset(
        torch.tensor(yd[input_columns].values,       dtype=torch.float32, device=device),
        X_sc, X_sc_strm, X_sc_res,
        torch.tensor(yd[source_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[delivery_columns].values,  dtype=torch.float32, device=device),
        torch.tensor(yd[stream_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[reservoir_columns].values, dtype=torch.float32, device=device),
        torch.tensor(yd['rchtype'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['hydseq'].values,     dtype=torch.float32, device=device),
        torch.tensor(yd['headflag'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['from_node'].values,  dtype=torch.int64,   device=device),
        torch.tensor(yd['to_node'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['frac'].values,       dtype=torch.float32, device=device),
        torch.tensor(yd['new_waterid'].values,dtype=torch.int64,   device=device),
        torch.tensor(yd['waterid'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['iftran'].values,     dtype=torch.int64,   device=device),
        obs,
    )
    yearly_loaders.append(DataLoader(ds, batch_size=len(yd), shuffle=False))


## Load spatial fold assignments and build index maps

Each fold is defined by a CSV file with one row per gauged reach.  Required columns:

| Column | Description |
|---|---|
| `waterid` | Reach ID (matches `sparrow_input.csv`) |
| `valsites` | 1 = held-out validation reach for this fold, 0 = training reach |

Set `cv_fold_dir` below to the folder containing your five fold files (`CV_1_data.csv` … `CV_5_data.csv`).

Index arrays that map into the `aligned_pred / aligned_obs` tensors returned by `gpu_align` are pre-built here so validation masking is O(1) inside the training loop.

In [ ]:
# Load validation waterids for each fold
# Set cv_fold_dir to the folder containing CV_1_data.csv … CV_5_data.csv
cv_fold_dir = 'data/5_fold_spatial/'   # <-- CHANGE THIS

fold_val_id_dict = {}
for fold in range(1, 6):
    df_fold = pd.read_csv(os.path.join(cv_fold_dir, f'CV_{fold}_data.csv'))
    fold_val_id_dict[fold] = set(
        df_fold.loc[df_fold['valsites'] == 1, 'waterid'].dropna().unique())
    print(f"Fold {fold}: {len(fold_val_id_dict[fold])} validation reaches")

# Build train_idx / val_idx relative to the aligned_obs vector
# (these indices are the same for every year-batch since the gauged set is constant)
for year, loader in zip(years, yearly_loaders):
    for batch in loader:
        batch = [b.to(device) for b in batch]
        obs_batch = batch[-1]
        ocid_batch = batch[-3]   # original_catchment_id
        mask_obs = ~torch.isnan(obs_batch)
        valid_idx  = torch.where(mask_obs)[0]
        valid_cids = ocid_batch[valid_idx].cpu().numpy()
        break
    break   # Only need one year to establish the index mapping

fold_indices = {}
for fold in range(1, 6):
    val_ids = fold_val_id_dict[fold]
    val_idx   = torch.tensor([i.item() for i, cid in zip(valid_idx, valid_cids)
                               if cid in val_ids],   device=device)
    train_idx = torch.tensor([i.item() for i, cid in zip(valid_idx, valid_cids)
                               if cid not in val_ids], device=device)
    fold_indices[fold] = {'train_idx': train_idx, 'val_idx': val_idx}
    print(f"Fold {fold}: train={train_idx.shape[0]} | val={val_idx.shape[0]}")


## 5-fold spatial cross-validation

In [ ]:
save_dir  = Path(output_dir)
criterion = nn.MSELoss()
all_fold_results = []

for fold in range(1, 6):
    print(f"\n=== Spatial Fold {fold}/5 ===")
    train_idx = fold_indices[fold]['train_idx']
    val_idx   = fold_indices[fold]['val_idx']

    param_model = ParamGenerator(
        input_size=len(input_columns), hidden_size=32,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    param_model_strm = ParamGenerator(
        input_size=len(input_columns_strm), hidden_size=8,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    param_model_res = ParamGenerator(
        input_size=len(input_columns_res), hidden_size=8,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    sparrow_model = SPARROW().to(device)

    optimizer = optim.Adam(
        list(param_model.parameters())
        + list(param_model_strm.parameters())
        + list(param_model_res.parameters()),
        lr=1e-3, weight_decay=1e-4)
    scheduler = SequentialLR(
        optimizer,
        schedulers=[LinearLR(optimizer, start_factor=0.1, total_iters=10),
                    StepLR(optimizer, step_size=50, gamma=0.5)],
        milestones=[10])

    fold_train_losses, fold_val_losses = [], []
    fold_train_r2_log, fold_val_r2_log, fold_val_r2_lin = [], [], []
    best_val = float('inf')

    for epoch in range(NUM_EPOCHS):
        param_model.train(); param_model_strm.train(); param_model_res.train()
        total_train = total_val = 0.0
        n_batches = 0
        ep_train_pred, ep_train_obs = [], []
        ep_val_pred,   ep_val_obs   = [], []

        shuffled = yearly_loaders.copy()
        random.shuffle(shuffled)

        for loader in shuffled:
            for batch in loader:
                (X_raw, X_sc, X_sc_strm, X_sc_res,
                 S, Z_D, Z_S, Z_R,
                 rtype, hseq, hflag, fnode, tnode, frac,
                 cid, ocid, iftran, obs) = [b.to(device) for b in batch]

                coeffs      = param_model(X_sc)
                coeffs_strm = param_model_strm(X_sc_strm)
                coeffs_res  = param_model_res(X_sc_res)

                alpha   = coeffs[:, :ns]
                theta_D = coeffs[:, ns:ns + nd]
                theta_S = coeffs_strm[:, -2:-1]
                theta_R = coeffs_res[:, -1:]

                _, _, _, total_load, ocid_out = sparrow_model(
                    S, Z_D, Z_S, Z_R, rtype, hseq, hflag, fnode, tnode,
                    alpha, theta_D, theta_S, theta_R,
                    frac, cid, ocid, dlvdsgn, iftran)

                total_load = total_load.to(device)
                aligned_pred, aligned_obs, mask = gpu_align(
                    total_load, obs, ocid_out, ocid)

                if mask.sum() > 0:
                    aligned_pred = aligned_pred.to(device)
                    aligned_obs  = aligned_obs.to(device)

                    # Training loss: train stations only
                    if train_idx.numel() > 0:
                        loss_train = criterion(
                            torch.log(aligned_pred[train_idx]),
                            torch.log(aligned_obs[train_idx]))
                        optimizer.zero_grad()
                        loss_train.backward()
                        optimizer.step()
                        total_train += loss_train.item()
                        ep_train_pred.append(aligned_pred[train_idx].detach().cpu())
                        ep_train_obs.append(aligned_obs[train_idx].detach().cpu())

                    # Validation loss: held-out stations
                    if val_idx.numel() > 0:
                        with torch.no_grad():
                            loss_val = criterion(
                                torch.log(aligned_pred[val_idx]),
                                torch.log(aligned_obs[val_idx]))
                            total_val += loss_val.item()
                            ep_val_pred.append(aligned_pred[val_idx].detach().cpu())
                            ep_val_obs.append(aligned_obs[val_idx].detach().cpu())

                    n_batches += 1

        avg_train = total_train / n_batches
        avg_val   = total_val   / n_batches
        fold_train_losses.append(avg_train)
        fold_val_losses.append(avg_val)

        # R² metrics
        r2_tr  = float(r2_logspace_torch(torch.cat(ep_train_obs), torch.cat(ep_train_pred))) if ep_train_pred else float('nan')
        r2_vl  = float(r2_logspace_torch(torch.cat(ep_val_obs),   torch.cat(ep_val_pred)))   if ep_val_pred   else float('nan')
        r2_vlin= float(r2_torch(torch.cat(ep_val_obs),            torch.cat(ep_val_pred)))   if ep_val_pred   else float('nan')
        fold_train_r2_log.append(r2_tr)
        fold_val_r2_log.append(r2_vl)
        fold_val_r2_lin.append(r2_vlin)

        # Checkpoint: save last and best
        ckpt = {'epoch': epoch+1,
                'param_model': param_model.state_dict(),
                'param_model_strm': param_model_strm.state_dict(),
                'param_model_res': param_model_res.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'val_loss': avg_val, 'r2_val_log': r2_vl}
        torch.save(ckpt, save_dir / f'fold{fold}_last.pt')
        if avg_val < best_val:
            best_val = avg_val
            torch.save(ckpt, save_dir / f'fold{fold}_best.pt')

        scheduler.step()
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
              f"R²(log): {r2_vl:.4f}")

    all_fold_results.append({
        'fold': fold,
        'train_losses': fold_train_losses, 'val_losses': fold_val_losses,
        'train_r2_log': fold_train_r2_log, 'val_r2_log':  fold_val_r2_log,
        'val_r2_linear': fold_val_r2_lin,
        'final_val_loss': fold_val_losses[-1],
    })

with open(os.path.join(output_dir, '5fold_spatial_results.json'), 'w') as f:
    json.dump({'folds': all_fold_results}, f, indent=2)
print("Results saved.")


## Summary

In [ ]:
final_r2  = [r['val_r2_log'][-1]  for r in all_fold_results]
final_r2l = [r['val_r2_linear'][-1] for r in all_fold_results]
print(f"Val R² (log-space) per fold: {[f'{v:.3f}' for v in final_r2]}")
print(f"  Mean ± SD: {np.mean(final_r2):.3f} ± {np.std(final_r2):.3f}")
print(f"Val R² (linear)   per fold: {[f'{v:.3f}' for v in final_r2l]}")
print(f"  Mean ± SD: {np.mean(final_r2l):.3f} ± {np.std(final_r2l):.3f}")
